# 05a — Sampling: NUTS without Hessian initialization (ANALYSIS)

Loads artifacts produced by `05a_sampling_no_hessian_data.ipynb` and the
Hessian saved by `04_training.ipynb`. No sampling here — only diagnostics,
figures, verdict.

**Structure**:
1. **Hessian** — eigenspectrum evolution across training + final spectrum
   at MLE, with an fp32 *resolvability* floor `τ = ε · max|λ|` overlaid
   (below this we cannot distinguish signal from float round-off in the
   chunked accumulation; we are not claiming a statistical noise model).
2. **Mass matrix** — eigenspectrum evolution across H1-1 adaptation rounds.
3. **Correlation** — per-round ACF heatmaps (lag × eigendirection) and ESS,
   showing how chain quality evolves with cumulative adaptation budget.

**Inputs**:
- `data/hessian.pt`
- `data/hessians_per_epoch.pt`
- `data/results_no_hessian/{RUN_NAME}/nuts_samples_no_hessian.pt`
- `data/results_no_hessian/{RUN_NAME}/iterative/round_NN.pt` (H1-1)

**Outputs** (into `data/results_no_hessian/{RUN_NAME}/`):
- `results.json` — H1 verdict + numerical summary
- `hessian_spectrum.{pdf,png}`, `m_spectrum_evolution.{pdf,png}`,
  `m_spectrum_trajectories.{pdf,png}`, `acf_per_round.{pdf,png}`,
  `ess_per_round.{pdf,png}`, `convergence_summary.{pdf,png}`


In [23]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


In [24]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root
if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
print(f"Data dir: {DATA_DIR}")


Data dir: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data


In [25]:
# ── Papermill parameters ──
RUN_NAME: str = "default"


## Load artifacts


In [26]:
RESULTS_DIR = DATA_DIR / "results_no_hessian" / RUN_NAME
ITER_DIR = RESULTS_DIR / "iterative"

# H1 production samples
payload = torch.load(RESULTS_DIR / "nuts_samples_no_hessian.pt", weights_only=False)
config = payload["config"]
N_WARMUP = config["n_warmup"]
N_SAMPLES = config["n_samples"]
SIGMA_PRIOR = config["sigma_prior"]
DGP_REGIME = config["dgp_regime"]
MAX_TREE_DEPTH = config["max_tree_depth"]

mle_param = payload["mle_param"]
adapted_M = payload["adapted_M"]
adapted_step_size = payload["adapted_step_size"]

samples = payload["chains"][0]["parameters"].float()
chain0_diag = payload["chains"][0]["diagnostics"]

print(f"Run: {RUN_NAME}")
print(f"  samples: {tuple(samples.shape)}  (N, d)")
print(f"  warmup: {N_WARMUP}, production: {N_SAMPLES}")
print(f"  accept: {payload['chains'][0]['acceptance_rate']:.3f}")
print(f"  step size: {adapted_step_size:.4e}")


Run: default
  samples: (500, 910)  (N, d)
  warmup: 2000, production: 500
  accept: 1.000
  step size: 2.1184e-03


In [27]:
# Hessian (computed once in 04_training.ipynb)
hess = torch.load(DATA_DIR / "hessian.pt", weights_only=True)
H_mle = hess["H_mle"].float()
d = int(H_mle.shape[0])
assert d == samples.shape[1], f"Dimension mismatch: H {d} vs samples {samples.shape[1]}"

eigvals, eigvecs = torch.linalg.eigh(H_mle)
# Threshold for non-degenerate directions (consistent with training nb)
eigval_threshold = eigvals.abs().max().item() * 1e-3
n_nd = int((eigvals.abs() > eigval_threshold).sum())
n_degen = d - n_nd

print(f"Hessian: λ ∈ [{eigvals[0]:.4e}, {eigvals[-1]:.4e}]")
print(f"  {n_nd} non-degenerate, {n_degen} degenerate directions")


Hessian: λ ∈ [-7.3382e-02, 3.9659e+00]
  158 non-degenerate, 752 degenerate directions


## H1-1: per-round artifacts

Load each `round_NN.pt`, compute mass-matrix spectrum, ACF in the Hessian
eigenbasis, and ESS per direction. All later sections (Mass matrix,
Correlation, Verdict) read from the `per_round` list built here.

Per-round budget is fixed (same `n_warmup_per_round` + `n_samples_per_round`
each round), but adaptation is cumulative — round `k` starts from the
adapted M of round `k−1`. So increasing round index ≈ increasing cumulative
adaptation budget.


In [ ]:
# ── Helpers: ACF (FFT, per Hessian eigendirection) and Geyer ESS ──
V = eigvecs.cpu().float()
sort_idx = torch.argsort(eigvals, descending=True)


def compute_acf_eigenbasis(samples_tensor, V, max_lag):
    """Project samples into Hessian eigenbasis, ACF per direction via FFT."""
    z = samples_tensor @ V
    d = z.shape[1]
    acf = torch.zeros(d, max_lag)
    for j in range(d):
        x = z[:, j]
        x = x - x.mean()
        var = x.var()
        if var < 1e-20:
            continue
        n = x.shape[0]
        padded = torch.zeros(2 * n)
        padded[:n] = x
        ft = torch.fft.rfft(padded)
        acov = torch.fft.irfft(ft * ft.conj())[:n] / n
        acf[j, :max_lag] = acov[:max_lag] / acov[0].clamp(min=1e-20)
    return acf


def geyer_ess(acf_row, n_draws):
    """Initial-positive-sequence ESS estimator (Geyer)."""
    total = 0.0
    for k in range(acf_row.shape[0]):
        if acf_row[k] < 0:
            break
        total += float(acf_row[k])
    tau = 1 + 2 * (total - 1)
    return n_draws / max(tau, 1.0)


In [ ]:
# ── Build per_round: load each round_NN.pt and compute spectra/ACF/ESS ──
round_paths = sorted(ITER_DIR.glob("round_*.pt")) if ITER_DIR.exists() else []

per_round = []
if round_paths:
    print(f"Found {len(round_paths)} H1-1 round checkpoints")
    for rp in round_paths:
        rd = torch.load(rp, weights_only=False)
        samps_r = rd["samples"].float()
        N_r = samps_r.shape[0]
        max_lag = min(500, max(N_r // 4, 1))

        acf_r = compute_acf_eigenbasis(samps_r, V, max_lag)[sort_idx]
        ess_r = torch.tensor(
            [geyer_ess(acf_r[i], N_r) for i in range(acf_r.shape[0])]
        )

        M_r = rd["mass_matrix"].detach().cpu().float()
        eigM_r = torch.sort(torch.linalg.eigvalsh(M_r), descending=True).values

        d_info = rd["diagnostics"]
        per_round.append(
            {
                "round": int(rd["round"]),
                "n_draws": N_r,
                "max_lag": max_lag,
                "samples": samps_r,
                "M_eigs": eigM_r,
                "acf": acf_r,  # (d, max_lag), sorted by Hessian eigenvalue descending
                "ess": ess_r,  # (d,)
                "step_size": float(d_info["step_size"]),
                "cond_M": float(d_info["cond_M"]),
                "acceptance_rate": float(d_info["acceptance_rate"]),
                "n_divergences": int(d_info["n_divergences"]),
                "mean_tree_depth": float(d_info["mean_tree_depth"]),
            }
        )

    print(f"Processed {len(per_round)} rounds:")
    for r in per_round:
        print(
            f"  round {r['round']:>2}  N={r['n_draws']}  "
            f"ε={r['step_size']:.2e}  cond(M)={r['cond_M']:.2e}  "
            f"accept={r['acceptance_rate']:.2f}"
        )
else:
    print("No H1-1 rounds found — downstream sections will be skipped.")


# Part 1 — Hessian

Eigenspectrum of the Hessian across training (from
`hessians_per_epoch.pt`) and at the final MLE. The dashed line marks the
**fp32 resolvability floor** `τ = ε · max|λ|` with `ε = 1.2e-7`: below `τ`
we cannot distinguish a true eigenvalue from float round-off in the
chunked accumulation that built `H`. This is a numerical floor, not a
statistical noise threshold.


In [ ]:
# ── Hessian eigenspectrum evolution across training + final spectrum ──
EPS_FP32 = 1.2e-7  # ≈ machine epsilon for float32

per_epoch_path = DATA_DIR / "hessians_per_epoch.pt"
have_evolution = per_epoch_path.exists()
if have_evolution:
    pe = torch.load(per_epoch_path, weights_only=True)
    epochs_arr = torch.tensor(pe["epochs"])
    eig_per_epoch = pe["eigvals_per_epoch"]  # (n_epochs, d)
    # Sort descending per epoch so column k is the k-th largest at each epoch
    eig_sorted_pe = torch.stack(
        [torch.sort(ev, descending=True).values for ev in eig_per_epoch]
    )
    # Per-epoch fp32 resolvability floor and rank boundaries where descending
    # eigenvalues cross ±τ. With eigenvalues sorted descending, sweeping rank
    # left → right we pass: λ > +τ (positive signal), |λ| < τ (below floor),
    # λ < −τ (negative signal). Two boundary curves on the heatmap.
    lam_max_per_epoch = eig_per_epoch.abs().max(dim=1).values
    tau_per_epoch = EPS_FP32 * lam_max_per_epoch  # (n_epochs,)
    n_epochs_e, d_e = eig_sorted_pe.shape
    below_pos = eig_sorted_pe < tau_per_epoch.unsqueeze(1)  # λ < +τ
    below_neg = eig_sorted_pe < -tau_per_epoch.unsqueeze(1)  # λ < −τ
    first_below_pos = torch.where(
        below_pos.any(dim=1),
        below_pos.float().argmax(dim=1),
        torch.full((n_epochs_e,), d_e),
    )
    first_below_neg = torch.where(
        below_neg.any(dim=1),
        below_neg.float().argmax(dim=1),
        torch.full((n_epochs_e,), d_e),
    )

eig_final_sorted = torch.sort(eigvals, descending=True).values
lam_max_final = float(eigvals.abs().max())
tau_final = EPS_FP32 * lam_max_final

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(18, 6))

if have_evolution:
    eps_clip = 1e-12
    signed_log = torch.sign(eig_sorted_pe) * torch.log10(
        eig_sorted_pe.abs() + eps_clip
    )
    vmax_clip = 4.0
    im = ax_l.imshow(
        signed_log.numpy(),
        aspect="auto",
        cmap="RdBu_r",
        origin="upper",
        extent=[1, d_e, epochs_arr[-1].item(), epochs_arr[0].item()],
        vmin=-vmax_clip,
        vmax=vmax_clip,
    )
    # Two boundary curves: where λ crosses +τ (enter floor band from above)
    # and where λ crosses −τ (exit floor band into the negative signal tail).
    ax_l.plot(
        first_below_pos.numpy() + 1,
        epochs_arr.numpy(),
        color="black",
        lw=1.5,
        ls="--",
        label="λ = +τ",
    )
    ax_l.plot(
        first_below_neg.numpy() + 1,
        epochs_arr.numpy(),
        color="black",
        lw=1.5,
        ls=":",
        label="λ = −τ",
    )
    ax_l.set_xlabel("Eigenvalue rank (descending)")
    ax_l.set_ylabel("Epoch")
    ax_l.set_title("Hessian eigenspectrum evolution\n(sign(λ)·log₁₀|λ|, clipped to ±4)")
    ax_l.legend(loc="upper right", title="fp32 floor (τ = ε·max|λ|)")
    plt.colorbar(im, ax=ax_l, label="sign(λ)·log₁₀|λ|")
else:
    ax_l.text(
        0.5,
        0.5,
        "hessians_per_epoch.pt not found\n(run 04_training.ipynb)",
        ha="center",
        va="center",
        transform=ax_l.transAxes,
    )
    ax_l.set_axis_off()

# Final Hessian spectrum at MLE — same encoding as the heatmap colorbar.
rank_idx = torch.arange(1, d + 1)
signed_log_final = torch.sign(eig_final_sorted) * torch.log10(
    eig_final_sorted.abs() + 1e-12
)
log_tau = float(torch.log10(torch.tensor(tau_final)))
ax_r.plot(
    rank_idx.numpy(),
    signed_log_final.numpy(),
    color="darkorange",
    lw=1.5,
    label="eig(H) at MLE",
)
ax_r.axhline(log_tau, color="black", lw=1.0, ls="--", label=f"+τ ({tau_final:.2e})")
ax_r.axhline(-log_tau, color="black", lw=1.0, ls=":", label="−τ")
ax_r.axhline(0, color="gray", lw=0.6, ls=":")
ax_r.set_xlabel("Eigenvalue rank (descending)")
ax_r.set_ylabel("sign(λ)·log₁₀|λ|")
n_above = int((eig_final_sorted.abs() > tau_final).sum())
ax_r.set_title(
    f"Final H spectrum  (above τ: {n_above}/{d}; "
    f"λ ∈ [{eig_final_sorted[-1]:.2e}, {eig_final_sorted[0]:.2e}])"
)
ax_r.grid(True, alpha=0.3)
ax_r.legend()

fig.suptitle("Hessian: training evolution + final state", fontweight="bold")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "hessian_spectrum.pdf", bbox_inches="tight", dpi=150)
fig.savefig(RESULTS_DIR / "hessian_spectrum.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"Final: τ = ε·max|λ| = {tau_final:.3e}")
print(f"       eigenvalues with |λ| > τ: {n_above}/{d}")


# Part 2 — Mass matrix

Raw eigenspectrum of the adapted mass matrix as cumulative adaptation
progresses across H1-1 rounds. No Hessian product, no clipping — just
`eig(M_round)` per round.


In [ ]:
# ── M spectrum evolution: heatmap (round × rank, color = log10 eig(M)) ──
if per_round:
    rounds = [r["round"] for r in per_round]
    n_rounds = len(rounds)
    M_sorted = torch.stack([r["M_eigs"] for r in per_round])  # (n_rounds, d)
    log_M = torch.log10(M_sorted.clamp(min=1e-30))
    finite_mask = torch.isfinite(log_M)
    vmin = float(log_M[finite_mask].min())
    vmax = float(log_M[finite_mask].max())

    fig, ax = plt.subplots(figsize=(max(6, 0.6 * n_rounds + 4), 7))
    im = ax.imshow(
        log_M.T.numpy(),
        aspect="auto",
        cmap="viridis",
        origin="upper",
        extent=[rounds[0] - 0.5, rounds[-1] + 0.5, d, 1],
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("H1-1 round  (cumulative adaptation)")
    ax.set_ylabel("Eigenvalue rank (descending, 1 = largest)")
    ax.set_title("Mass matrix eigenspectrum evolution  (log₁₀ eig(M))", fontweight="bold")
    ax.set_xticks(rounds)
    plt.colorbar(im, ax=ax, label="log₁₀ eig(M)")
    fig.tight_layout()
    fig.savefig(ITER_DIR / "m_spectrum_evolution.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "m_spectrum_evolution.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  log₁₀ eig(M) ∈ [{vmin:.2f}, {vmax:.2f}]")
else:
    print("Skipping: no per-round data.")


In [ ]:
# ── M spectrum: per-rank trajectories across rounds ──
if per_round:
    fig, ax = plt.subplots(figsize=(10, 5))
    sample_ranks = sorted(
        set(
            r
            for r in [0, d // 8, d // 4, d // 2, 3 * d // 4, 7 * d // 8, d - 1]
            if 0 <= r < d
        )
    )
    for r in sample_ranks:
        ax.plot(
            rounds,
            M_sorted[:, r].clamp(min=1e-30).numpy(),
            "o-",
            lw=1.5,
            label=f"rank {r}",
        )
    ax.set_yscale("log")
    ax.set_xlabel("H1-1 round")
    ax.set_ylabel("eig(M) at fixed rank")
    ax.set_title("Mass matrix eigenvalues per rank across rounds")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(ITER_DIR / "m_spectrum_trajectories.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "m_spectrum_trajectories.png", bbox_inches="tight", dpi=150)
    plt.show()


# Part 3 — Correlation

Per-round chain quality. Each row corresponds to a fixed point in the
adaptation curve (round index = cumulative adaptation budget). Within a
row, the production chain has a fixed length `n_draws`, so the x-axis on
ACF panels is *lag* (not draw count).

The `n_nd` non-degenerate Hessian directions (`|λ(H)| > τ`) sit at the top
of each panel; the degenerate tail below is where mixing is hardest.


In [ ]:
# ── Non-degenerate / degenerate Hessian split (used as boundary in panels) ──
n_nd = int((eigvals.abs() > tau_final).sum())
n_degen = d - n_nd
print(f"Hessian split at τ = {tau_final:.2e}: {n_nd} non-degenerate, {n_degen} degenerate")


In [ ]:
# ── ACF heatmap per round (one row per round) ──
if per_round:
    n_rounds = len(per_round)
    fig, axes = plt.subplots(
        n_rounds, 1, figsize=(14, 2.5 * n_rounds + 1), sharex=False, squeeze=False
    )
    for i, r in enumerate(per_round):
        ax = axes[i, 0]
        im = ax.imshow(
            r["acf"].numpy(),
            aspect="auto",
            cmap="RdBu_r",
            vmin=-0.3,
            vmax=1.0,
            interpolation="nearest",
            origin="upper",
        )
        if 0 < n_nd < d:
            ax.axhline(n_nd - 0.5, color="lime", lw=1.5, ls="--")
        ax.set_ylabel(
            f"round {r['round']}\nN={r['n_draws']}", fontsize=9
        )
        if i == n_rounds - 1:
            ax.set_xlabel("Lag τ")
        ax.set_title(
            f"ε={r['step_size']:.2e}  cond(M)={r['cond_M']:.2e}  "
            f"accept={r['acceptance_rate']:.2f}",
            fontsize=9,
        )
    fig.suptitle(
        "ACF per Hessian eigendirection — across H1-1 rounds  "
        "(green dashed: non-degenerate boundary)",
        fontweight="bold",
        y=1.0,
    )
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, label="ACF")
    fig.savefig(ITER_DIR / "acf_per_round.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "acf_per_round.png", bbox_inches="tight", dpi=150)
    plt.show()


In [ ]:
# ── ESS per eigendirection — one panel per round ──
if per_round:
    n_rounds = len(per_round)
    fig, axes = plt.subplots(
        n_rounds, 1, figsize=(14, 1.8 * n_rounds + 1), sharex=True, squeeze=False
    )
    for i, r in enumerate(per_round):
        ax = axes[i, 0]
        ax.bar(
            range(d),
            r["ess"].numpy(),
            width=1.0,
            color="steelblue",
            alpha=0.7,
        )
        if 0 < n_nd < d:
            ax.axvline(n_nd - 0.5, color="lime", lw=1.5, ls="--")
        ax.set_ylabel(
            f"round {r['round']}\nN={r['n_draws']}", fontsize=9
        )
        ax.set_title(
            f"ESS median={r['ess'].median():.1f} / {r['n_draws']}  "
            f"(nd: {r['ess'][:n_nd].median():.1f}, dg: {r['ess'][n_nd:].median():.1f})",
            fontsize=9,
        )
        ax.set_xlim(-0.5, d - 0.5)
        ax.grid(True, alpha=0.3, axis="y")
    axes[-1, 0].set_xlabel("Eigendirection (sorted by λ(H), largest first)")
    fig.suptitle(
        "ESS per Hessian eigendirection — across H1-1 rounds",
        fontweight="bold",
        y=1.0,
    )
    fig.tight_layout()
    fig.savefig(ITER_DIR / "ess_per_round.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "ess_per_round.png", bbox_inches="tight", dpi=150)
    plt.show()


In [ ]:
# ── H1-1 convergence summary: per-round ACF@50, ESS, cond(M) ──
if per_round:
    summary = []
    for r in per_round:
        lag50 = min(50, r["max_lag"] - 1)
        acf50 = r["acf"][:, lag50]
        summary.append(
            {
                "round": r["round"],
                "n_draws": r["n_draws"],
                "step_size": r["step_size"],
                "cond_M": r["cond_M"],
                "acceptance_rate": r["acceptance_rate"],
                "n_divergences": r["n_divergences"],
                "mean_tree_depth": r["mean_tree_depth"],
                "acf50_median": float(acf50.median()),
                "acf50_nd_median": float(acf50[:n_nd].median()),
                "acf50_dg_median": float(acf50[n_nd:].median()),
                "ess_median": float(r["ess"].median()),
                "ess_nd_median": float(r["ess"][:n_nd].median()),
                "ess_dg_median": float(r["ess"][n_nd:].median()),
            }
        )

    rounds = [s["round"] for s in summary]
    acf50s = [s["acf50_median"] for s in summary]
    acf50_nds = [s["acf50_nd_median"] for s in summary]
    ess_meds = [s["ess_median"] for s in summary]
    ess_nd_meds = [s["ess_nd_median"] for s in summary]
    conds = [s["cond_M"] for s in summary]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    ax = axes[0]
    ax.plot(rounds, acf50s, "o-", color="teal", lw=2, label="all")
    ax.plot(rounds, acf50_nds, "s--", color="darkorange", lw=1.5, label="non-degen")
    ax.axhline(0.5, color="red", ls=":", lw=1, label="H1 threshold")
    ax.axhline(0.3, color="green", ls=":", lw=1, label="falsified threshold")
    ax.set_xlabel("Round")
    ax.set_ylabel("ACF@50 median")
    ax.set_title("ACF@50 across rounds")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

    ax = axes[1]
    ax.plot(rounds, ess_meds, "o-", color="steelblue", lw=2, label="all")
    ax.plot(rounds, ess_nd_meds, "s--", color="darkorange", lw=1.5, label="non-degen")
    ax.set_xlabel("Round")
    ax.set_ylabel("ESS median")
    ax.set_title("ESS across rounds")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(rounds, conds, "o-", color="crimson", lw=2)
    ax.set_xlabel("Round")
    ax.set_ylabel("cond(M)")
    ax.set_yscale("log")
    ax.set_title("Mass matrix conditioning")
    ax.grid(True, alpha=0.3)

    fig.suptitle("H1-1: adaptation convergence", fontweight="bold")
    fig.tight_layout()
    fig.savefig(ITER_DIR / "convergence_summary.pdf", bbox_inches="tight", dpi=150)
    fig.savefig(ITER_DIR / "convergence_summary.png", bbox_inches="tight", dpi=150)
    plt.show()

    with open(ITER_DIR / "round_summary.json", "w") as f:
        import json as _json
        _json.dump(summary, f, indent=2)

    print(
        f"\n{'Round':<6} {'ACF@50':<8} {'ACF@50(nd)':<11} {'ESS':<8} "
        f"{'ESS(nd)':<9} {'cond(M)':<10}"
    )
    print("-" * 58)
    for s in summary:
        print(
            f"{s['round']:<6} {s['acf50_median']:<8.3f} {s['acf50_nd_median']:<11.3f} "
            f"{s['ess_median']:<8.1f} {s['ess_nd_median']:<9.1f} {s['cond_M']:<10.2e}"
        )


## Verdict

Hypothesis H1: *singular geometry defeats any fixed mass matrix.* The
verdict is taken from the **final H1-1 round** (most-adapted state) — that
is the strongest test of whether adaptation can rescue NUTS without
Hessian information.

- **ACF@50 median < 0.3** → falsified (good mixing achieved by adaptation)
- **ACF@50 median > 0.5** → confirmed (singular geometry defeats adaptation)
- in between → inconclusive


In [ ]:
import json as _json

if per_round:
    final = summary[-1]
    first = summary[0]
    acf50_final = final["acf50_median"]
    ess_final = final["ess_median"]

    if acf50_final < 0.3:
        verdict = "FALSIFIED"
    elif acf50_final > 0.5:
        verdict = "CONFIRMED"
    else:
        verdict = "INCONCLUSIVE"

    ess_gain = ess_final / max(first["ess_median"], 0.1)

    results = {
        "experiment": "05a_sampling_no_hessian",
        "run_name": RUN_NAME,
        "hypothesis": "H1: singular geometry defeats any fixed mass matrix",
        "verdict": verdict,
        "verdict_source": "final H1-1 round",
        "config": {
            "d": d,
            "n_warmup": N_WARMUP,
            "n_samples": N_SAMPLES,
            "max_tree_depth": MAX_TREE_DEPTH,
            "sigma_prior": SIGMA_PRIOR,
            "dgp_regime": DGP_REGIME,
        },
        "hessian": {
            "tau_fp32": tau_final,
            "n_nondegen": n_nd,
            "n_degen": n_degen,
            "eig_range": [float(eig_final_sorted[-1]), float(eig_final_sorted[0])],
        },
        "h11_final": {
            "round": final["round"],
            "n_draws": final["n_draws"],
            "acf50_median": acf50_final,
            "acf50_nd_median": final["acf50_nd_median"],
            "acf50_dg_median": final["acf50_dg_median"],
            "ess_median": ess_final,
            "ess_nd_median": final["ess_nd_median"],
            "ess_dg_median": final["ess_dg_median"],
            "cond_M": final["cond_M"],
            "step_size": final["step_size"],
            "acceptance_rate": final["acceptance_rate"],
        },
        "h11_first": {
            "round": first["round"],
            "acf50_median": first["acf50_median"],
            "ess_median": first["ess_median"],
        },
        "h11_ess_gain": ess_gain,
    }

    with open(RESULTS_DIR / "results.json", "w") as f:
        _json.dump(results, f, indent=2)

    print(f"★ H1 VERDICT: {verdict}  (from final round {final['round']})")
    print(f"  ACF@50 median = {acf50_final:.3f}")
    print(f"  ESS median = {ess_final:.1f} / {final['n_draws']}  (gain {ess_gain:.2f}×)")
    print(f"  Results saved to {RESULTS_DIR / 'results.json'}")
else:
    print("No H1-1 rounds — cannot compute verdict.")
